## Lab 7: Model Registry, Model Serving, and Model Monitoring

-   **Course:** Engineering of Intelligent Models
-   **Module:** M4. Operations (Deployment & Monitoring)
-   **Focus:** MLflow Model Registry, FastAPI Model Serving, Evidently AI Model Monitoring
-   **Branch:** `Lab7`

### 1\. Goal of the Laboratory
Up to this point, our models have been evaluated statically. However, an MLOps pipeline is only as valuable as the predictions it serves and its resilience to change. Over time, the statistical properties of the independent variables (X) can change, leading to a phenomenon known as **Data Drift**. When this occurs, the model's performance inevitably degrades. This, of course, leads to **Model Drift**, where the model's predictions become less accurate over time, necessitating retraining or adjustments.

In this laboratory, we will:
1.  **Govern our Artifacts:** Use MLflow to formally register our best model and transition it to the `Production` stage.
2.  **Serve the Model:** Implement a FastAPI service that dynamically loads the `Production` model from the registry to serve real-time HTTP requests.
3.  **Monitor Health:** Integrate Evidently AI as a dedicated service, artificially inject a weather anomaly (Data Drift), and generate a statistical report to trigger our monitoring alarms.

### 2\.1 Model Registry (MLflow Model Registry)
The MLflow Model Registry is a centralized repository for managing the lifecycle of machine learning models. It provides a structured way to track, version, and manage models, making it easier to deploy and maintain them in production. The Model Registry allows us to:
-   **Register Models:** Store models with metadata, such as version, stage (e.g., `Staging`, `Production`), and description.
-   **Transition Stages:** Move models through different stages of the lifecycle (e.g., from `Staging` to `Production`) to manage their deployment status.
-   **Track Lineage:** Keep track of the lineage of models, including the data and code used to create them, which is crucial for reproducibility and auditing.

<img src="imagens/mlflow-model-registry.png" alt="MLflow Model Registry" width="600"/>

In this Lab, we will use the MLflow Model Registry to register our best-performing model from Lab 6 and transition it to the `Production` stage. This will allow us to serve the model in real-time using FastAPI and monitor its performance over time with Evidently AI.

However, we didn't tag any of our models to `Staging` or `Production` in Lab 6, so for demonstration purposes, we will manually select the best model from the MLflow UI and transition it to `Production`. In a real-world scenario, you would typically automate this process based on performance metrics or other criteria.

### 2\.2 Model Serving (FastAPI)
FastAPI is a modern, fast (high-performance) web framework for building APIs with Python. It is designed to be easy to use and allows for the rapid development of APIs. FastAPI is particularly well-suited for serving machine learning models due to its asynchronous capabilities and support for data validation. With FastAPI, we can create an API endpoint that loads the `Production` model from the MLflow Model Registry and serves predictions in real-time. This allows us to integrate our model into applications and provide predictions to end-users or other systems.

<img src="imagens/model-serving-fastapi.png" alt="Model Serving with FastAPI" width="600"/>

In other words:
- We will save our best model from Lab 6 to the MLflow Model Registry and transition it to `Production`.
- We will implement a FastAPI service that dynamically loads the `Production` model from the registry.
- The FastAPI service will expose an endpoint that accepts input data, processes it, and returns predictions in real-time.

### 2\.3 Data Drift vs. Model Drift
Before we dive into the technical implementation, let's clarify two critical concepts:
-   **Data Drift:** Refers to changes in the input data distribution over time. For example, if our model was trained on data collected during normal weather conditions, and suddenly we encounter an extreme weather event (e.g., a hurricane), the statistical properties of the input features may change significantly, leading to degraded model performance.
-   **Model Drift:** Refers to the degradation of model performance over time, which can be caused by data drift, changes in the underlying relationships between features and target variable, or even changes in the environment where the model is deployed.

<img src="imagens/data_drift.webp" alt="Data Drift vs Model Drift" width="600"/>

In order to maintain the reliability of our ML models in production, it is crucial to monitor model performance and detect any signs of model drift, since data drift is almost inevitable in real-world applications. By implementing robust monitoring and alerting mechanisms, we can proactively address issues related to data drift and model drift, ensuring that our models continue to deliver accurate predictions over time.

<img src="imagens/model_decay_retraining.png" alt="Model Decay and Retraining" width="600"/>

In this lab, we will focus on detecting data drift using Evidently AI, which will help us identify when the input data distribution has changed significantly, allowing us to take appropriate actions such as retraining the model or adjusting our monitoring thresholds.

### 3\. Infrastructure Expansion (Docker Compose)
Before writing Python code, we must expand our infrastructure to include our new microservices: the Inference API and the Monitoring UI.

##### Step 1: Commit changes and change to Lab7
Save your Lab 6 progress and create an isolated environment for Lab 7:

In [ ]:
!git add .
!git commit -m "Lab6: Complete daily evaluation and artifact logging"
!git checkout -b Lab7

##### Step 1: Setup the API Directory Structure
In a mature MLOps ecosystem, the environment used to _train_ a model must be strictly isolated from the environments used to _serve_ and _monitor_ it. Monolithic architectures lead to dependency conflicts and security vulnerabilities.

To achieve a production-grade setup, we will create two independent build contexts: one for our FastAPI inference service, and one for our Evidently AI monitoring dashboard. Both will utilize a lightweight Python base image to minimize resource overhead.

Create two new root directories to house the configurations for our new microservices:

In [8]:
# Do not forget to change root folder if you're running this notebook from my official repository
import os
os.chdir('../../') # Change to the root of the repository

# Then, create the necessary directories for the API service
!mkdir api
!mkdir api\app
!mkdir evidently_ui\workspace

A subdirectory or file api already exists.
A subdirectory or file api\app already exists.


##### Step 2: Create the Dedicated API Dockerfile
Following the official FastAPI deployment standards, create `api/Dockerfile`. This uses a high-performance Python base image and utilizes the modern `fastapi run` CLI.

```dockerfile
# api/Dockerfile
FROM python:3.12-slim

WORKDIR /code

# Install system dependencies if needed (e.g., for certain ML libraries)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

COPY ./api/requirements.txt /code/requirements.txt

RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt

# Copy the application code into the /code/app directory
COPY ./api/app /code/app

# Start the FastAPI application using the production-ready CLI
CMD ["fastapi", "run", "app/main.py", "--port", "80"]
```

##### Step 3: Update `docker-compose.yaml`
Add the following two services to the bottom of your `docker-compose.yaml` file. We will build the Dockerfile in `api/` for FastAPI and the official Evidently UI image for monitoring.

```yaml
  fastapi-service:
    container_name: emi-fastapi
    build:
      context: .
      dockerfile: api/Dockerfile
    depends_on:
      - mlflow_server
    restart: unless-stopped
    ports:
      - "8585:80"
    environment:
      - MLFLOW_TRACKING_URI=http://mlflow_server:5000
    volumes:
      - ./api/app:/code/app # Hot-reloading for development

  evidently-ui:
    container_name: emi-evidently
    image: ghcr.io/evidentlyai/evidently:latest
    depends_on:
      - mlflow_server
    ports:
      - "8081:8000" # Mapped to 8081 to avoid conflict with FastAPI
    volumes:
      - ./evidently_workspace:/app/workspace
```

##### Step 4: Create API Requirements
This file should be distinct from your main `requirements.txt`. It only needs the essentials for inference:
```text
fastapi[standard]==1.0.3
mlflow==3.10.1
torch==2.10.0
numpy==2.4.3
pandas==3.0.1
```